# Stage 3: GRPO Reasoning (Thought + Action)

GRPO с форматом **текст + действие** поверх checkpoint из Stage 2 (`grpo_curriculum`).

- **Init:** веса из GRPO curriculum (8→10→12→16), fallback — SFT curriculum
- **Формат вывода:** одно предложение `Thought:` + `Action: <0|1|2>`
- **Train:** тот же curriculum 8x8 → 10x10 → 12x12 → 16x16
- **Compare:** sample efficiency и final quality vs GRPO action-only (Stage 2)

### Формат промпта

```
Thought: I am facing east, goal is ahead and to the right. I should move forward.
Action: 2
```

1. **Одно предложение Thought** — даёт reasoning signal, но не перегружает nanoVLM-222M.
2. **Разделитель `Action:`** — надёжный парсинг действия из сгенерированного текста.
3. **Ситуация + план** — учит модель формировать внутреннее представление позиции/направления, что критично на 16x16 (~30 шагов).

In [1]:
!pip install -q gymnasium minigrid torch torchvision pillow tqdm matplotlib transformers huggingface-hub safetensors
!test -d nanoVLM || git clone --branch v0.1 --depth 1 https://github.com/huggingface/nanoVLM.git

import sys
sys.path.insert(0, "nanoVLM")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 require

In [3]:
import json
import random
import re
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import minigrid
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from minigrid.envs.empty import EmptyEnv
from torch.distributions import Categorical
from tqdm.auto import tqdm

from data.processors import get_image_processor, get_tokenizer
from models.config import VLMConfig
from models.vision_language_model import VisionLanguageModel

for size in [10, 12, 14]:
    env_id = f"MiniGrid-Empty-{size}x{size}-v0"
    if env_id not in gym.envs.registry:
        gym.register(id=env_id, entry_point=EmptyEnv, kwargs={"size": size})

GRPO_CURRICULUM_CANDIDATES = [
    Path("/kaggle/input/datasets/gleror/grpo-curriculum-res"),
    Path("grpo_curriculum_output"),
]
SFT_DATASET_CANDIDATES = [
    Path("/kaggle/input/sft-curriculum-final"),
    Path("output"),
]

GRPO_CURRICULUM_PATH = next(
    (p for p in GRPO_CURRICULUM_CANDIDATES if (p / "checkpoint").exists()), None
)
SFT_DATASET_PATH = next(
    (p for p in SFT_DATASET_CANDIDATES if (p / "checkpoint").exists()), None
)

if GRPO_CURRICULUM_PATH is not None:
    INIT_CHECKPOINT = GRPO_CURRICULUM_PATH / "checkpoint"
    INIT_SOURCE = "grpo_curriculum"
    BASELINE_RESULTS_PATH = GRPO_CURRICULUM_PATH / "results.json"

OUTPUT_DIR = Path(
    "/kaggle/working/grpo_reasoning" if Path("/kaggle").exists() else "grpo_reasoning_output"
)

STAGES = [
    {"env": "MiniGrid-Empty-8x8-v0",   "iters": 5, "max_steps": 50},
    {"env": "MiniGrid-Empty-10x10-v0", "iters": 5, "max_steps": 70},
    {"env": "MiniGrid-Empty-12x12-v0", "iters": 5, "max_steps": 90},
    {"env": "MiniGrid-Empty-16x16-v0", "iters": 5, "max_steps": 110},
]
N_ITERATIONS = sum(s["iters"] for s in STAGES)

EVAL_ENV = "MiniGrid-Empty-16x16-v0"
EVAL_ENVS = ["MiniGrid-Empty-8x8-v0", "MiniGrid-Empty-16x16-v0"]
FINAL_EVAL_ENVS = [
    "MiniGrid-Empty-8x8-v0",
    "MiniGrid-Empty-10x10-v0",
    "MiniGrid-Empty-12x12-v0",
    "MiniGrid-Empty-16x16-v0",
]

PROMPT = (
    "Question: You are a navigation agent in a grid world. "
    "Actions: 0=left, 1=right, 2=forward. "
    "First describe your situation briefly in one sentence, then choose action. "
    "Answer:\n"
)
ACTION_PATTERN = re.compile(r"Action:\s*([012])", re.IGNORECASE)

K_GROUP = 8
EVAL_MAX_STEPS = 150
LR = 1e-5
BETA_KL = 0.01
STEP_PENALTY = 0.01
DISTANCE_REWARD_SCALE = 0.5
PER_STEP_PROGRESS_SCALE = 0.1
TEMPERATURE = 1.5
MAX_RESPONSE_TOKENS = 48
SUCCESS_THRESHOLD = 0.5
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = get_tokenizer(VLMConfig.lm_tokenizer)
image_processor = get_image_processor(VLMConfig.vit_img_size)

print(f"Device: {device}")
print(f"Init checkpoint: {INIT_CHECKPOINT} ({INIT_SOURCE})")
print(f"Curriculum: {N_ITERATIONS} total iters across {len(STAGES)} stages")
for i, s in enumerate(STAGES):
    label = s["env"].replace("MiniGrid-Empty-", "").replace("-v0", "")
    print(f"  Stage {i + 1}: {label} — {s['iters']} iters, max_steps={s['max_steps']}")

Device: cuda
Init checkpoint: /kaggle/input/datasets/gleror/grpo-curriculum-res/checkpoint (grpo_curriculum)
Curriculum: 20 total iters across 4 stages
  Stage 1: 8x8 — 5 iters, max_steps=50
  Stage 2: 10x10 — 5 iters, max_steps=70
  Stage 3: 12x12 — 5 iters, max_steps=90
  Stage 4: 16x16 — 5 iters, max_steps=110


## 2. Load checkpoint

`model` — обучаемая политика. `ref_model` — замороженная копия init checkpoint на CPU для KL penalty.

In [4]:
print(f"Loading init checkpoint ({INIT_SOURCE})...")
model = VisionLanguageModel.from_pretrained(str(INIT_CHECKPOINT)).to(device)
model.train()

ref_model = VisionLanguageModel.from_pretrained(str(INIT_CHECKPOINT)).to("cpu")
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

grpo_action_baseline = None
if BASELINE_RESULTS_PATH and BASELINE_RESULTS_PATH.exists():
    with open(BASELINE_RESULTS_PATH) as f:
        grpo_action_baseline = json.load(f)
    print(f"Loaded Stage 2 baseline: {BASELINE_RESULTS_PATH}")
    if EVAL_ENV in grpo_action_baseline.get("eval", {}):
        m = grpo_action_baseline["eval"][EVAL_ENV]
        print(f"  GRPO action-only 16x16: success={m['success_rate']:.1%}")

Loading init checkpoint (grpo_curriculum)...
Loaded Stage 2 baseline: /kaggle/input/datasets/gleror/grpo-curriculum-res/results.json
  GRPO action-only 16x16: success=100.0%


## 3. Policy helpers (Thought + Action)

Генерация: autoregressive sampling до `Action: <digit>` или `MAX_RESPONSE_TOKENS`.
Loss: сумма log π по всей сгенерированной последовательности.

In [5]:
def get_full_logits(model, input_ids, image, attention_mask=None):
    """Logits всего vocab на каждой текстовой позиции [batch, seq, vocab]."""
    image_embd = model.vision_encoder(image)
    image_embd = model.MP(image_embd)
    token_embd = model.decoder.token_embedding(input_ids)
    combined_embd = torch.cat((image_embd, token_embd), dim=1)

    if attention_mask is not None:
        img_mask = torch.ones(
            image_embd.size(0), image_embd.size(1),
            device=attention_mask.device, dtype=attention_mask.dtype,
        )
        attention_mask = torch.cat((img_mask, attention_mask), dim=1)

    hidden = model.decoder(combined_embd, attention_mask)
    logits = model.decoder.head(hidden)
    return logits[:, image_embd.size(1):, :]


def get_last_logits(model, input_ids, image, attention_mask=None):
    """Logits на последней позиции (для autoregressive generation)."""
    return get_full_logits(model, input_ids, image, attention_mask)[:, -1, :]


def parse_action_from_response(text):
    """Извлечь действие из формата Thought: ... Action: X."""
    match = ACTION_PATTERN.search(text)
    if match:
        return int(match.group(1))
    for ch in reversed(text):
        if ch in "012":
            return int(ch)
    return 2


def has_valid_action(text):
    return ACTION_PATTERN.search(text) is not None


def _append_token(input_ids, attention_mask, token_id):
    new_id = torch.tensor([[token_id]], device=input_ids.device, dtype=input_ids.dtype)
    new_mask = torch.ones(1, 1, device=attention_mask.device, dtype=attention_mask.dtype)
    return (
        torch.cat([input_ids, new_id], dim=1),
        torch.cat([attention_mask, new_mask], dim=1),
    )


@torch.no_grad()
def generate_response(model, prompt_ids, image, prompt_mask, temperature=TEMPERATURE, greedy=False):
    """Сэмплировать Thought + Action. Возвращает response_ids, text, action, valid_format."""
    input_ids = prompt_ids.clone()
    attention_mask = prompt_mask.clone()
    generated = []

    for _ in range(MAX_RESPONSE_TOKENS):
        logits = get_last_logits(model, input_ids, image, attention_mask)
        if greedy:
            token_id = logits.argmax(dim=-1).item()
        else:
            token_id = Categorical(logits=logits / temperature).sample().item()
        generated.append(token_id)
        input_ids, attention_mask = _append_token(input_ids, attention_mask, token_id)

        text = tokenizer.decode(generated, skip_special_tokens=True)
        if has_valid_action(text):
            break

    text = tokenizer.decode(generated, skip_special_tokens=True)
    action = parse_action_from_response(text)
    response_ids = torch.tensor([generated], device=prompt_ids.device, dtype=prompt_ids.dtype)
    return response_ids, text, action, has_valid_action(text)


def sequence_log_probs_and_kl(model, ref_model, prompt_ids, response_ids, image, prompt_mask):
    """Сумма log π(response|prompt) и mean KL по позициям ответа."""
    full_ids = torch.cat([prompt_ids, response_ids], dim=1)
    resp_mask = torch.ones_like(response_ids, dtype=prompt_mask.dtype, device=prompt_mask.device)
    full_mask = torch.cat([prompt_mask, resp_mask], dim=1)

    logits = get_full_logits(model, full_ids, image, full_mask)
    prompt_len = prompt_ids.size(1)
    n_resp = response_ids.size(1)

    log_prob_sum = torch.zeros(1, device=logits.device)
    kl_sum = torch.zeros(1, device=logits.device)

    with torch.no_grad():
        ref_logits = get_full_logits(
            ref_model, full_ids.cpu(), image.cpu(), full_mask.cpu()
        ).to(logits.device)

    for i in range(n_resp):
        pos = prompt_len + i - 1
        tok_id = response_ids[0, i]
        step_logits = logits[0, pos]
        log_prob_sum = log_prob_sum + F.log_softmax(step_logits, dim=-1)[tok_id]

        p = F.softmax(step_logits, dim=-1)
        q = F.softmax(ref_logits[0, pos], dim=-1)
        kl_sum = kl_sum + (p * (p.log() - q.log())).sum()

    return log_prob_sum.squeeze(), (kl_sum / max(n_resp, 1)).squeeze()


@torch.no_grad()
def predict_action_greedy(model, env):
    """Greedy Thought+Action для eval."""
    model.eval()
    rgb = env.render()
    image = Image.fromarray(rgb).convert("RGB")
    processed = image_processor(image).unsqueeze(0).to(device)
    prompt_ids = tokenizer.encode(PROMPT, return_tensors="pt").to(device)
    prompt_mask = torch.ones_like(prompt_ids)
    _, _, action, _ = generate_response(model, prompt_ids, processed, prompt_mask, greedy=True)
    model.train()
    return action


def evaluate_policy(model, env_id, n_episodes=30, max_steps=200, seed=0, desc=None, leave=False):
    env = gym.make(env_id, render_mode="rgb_array")
    successes, returns, lengths = 0, [], []
    model.eval()

    label = env_label(env_id)
    ep_bar = tqdm(
        range(n_episodes),
        desc=desc or f"Eval {label}",
        leave=leave,
        unit="ep",
    )
    for ep in ep_bar:
        obs, _ = env.reset(seed=seed + ep)
        total_reward = 0.0
        for step in range(max_steps):
            action = predict_action_greedy(model, env)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                if terminated and reward > 0:
                    successes += 1
                lengths.append(step + 1)
                break
        else:
            lengths.append(max_steps)
        returns.append(total_reward)

        done = ep + 1
        ep_bar.set_postfix(
            succ=f"{successes / done:.0%}",
            ret=f"{np.mean(returns):.2f}",
            len=f"{np.mean(lengths):.1f}",
        )

    env.close()
    model.train()
    return {
        "success_rate": successes / n_episodes,
        "avg_return": float(np.mean(returns)),
        "avg_length": float(np.mean(lengths)),
    }


def env_label(env_id):
    return env_id.replace("MiniGrid-Empty-", "").replace("-v0", "")

## 4. Rollout + GRPO update

Reward: shaped distance + success − step penalty.
Дополнительно логируется `valid_format` и длина Thought для анализа.

In [6]:
def find_goal(env):
    for x in range(env.width):
        for y in range(env.height):
            cell = env.grid.get(x, y)
            if cell is not None and cell.type == "goal":
                return (x, y)
    raise RuntimeError("Goal not found")


def manhattan_dist(pos, goal):
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])


def compute_advantages(rewards):
    rewards = np.asarray(rewards, dtype=np.float32)
    std = rewards.std()
    if std > 1e-4:
        return (rewards - rewards.mean()) / (std + 1e-8)
    ranks = np.argsort(np.argsort(rewards)).astype(np.float32)
    return (ranks - ranks.mean()) / (ranks.std() + 1e-8)


@torch.no_grad()
def rollout_episode(env_id, max_steps, seed):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    unwrapped = env.unwrapped
    goal_pos = find_goal(unwrapped)
    initial_dist = manhattan_dist(unwrapped.agent_pos, goal_pos)

    steps = []
    success = False
    episode_reward = 0.0
    prev_dist = initial_dist
    valid_formats = 0

    for _ in range(max_steps):
        rgb = env.render()
        image = Image.fromarray(rgb).convert("RGB")
        processed = image_processor(image).unsqueeze(0).to(device)
        prompt_ids = tokenizer.encode(PROMPT, return_tensors="pt").to(device)
        prompt_mask = torch.ones_like(prompt_ids)

        response_ids, response_text, action, valid_fmt = generate_response(
            model, prompt_ids, processed, prompt_mask, temperature=TEMPERATURE
        )
        valid_formats += int(valid_fmt)

        obs, reward, terminated, truncated, _ = env.step(action)

        new_dist = manhattan_dist(unwrapped.agent_pos, goal_pos)
        episode_reward += PER_STEP_PROGRESS_SCALE * (prev_dist - new_dist)
        prev_dist = new_dist

        steps.append({
            "prompt_ids": prompt_ids.detach().cpu(),
            "prompt_mask": prompt_mask.detach().cpu(),
            "response_ids": response_ids.detach().cpu(),
            "image": processed.detach().cpu(),
            "action": action,
            "response_text": response_text,
            "valid_format": valid_fmt,
        })
        if terminated or truncated:
            success = terminated and reward > 0
            break

    final_dist = manhattan_dist(unwrapped.agent_pos, goal_pos)
    progress = (initial_dist - final_dist) / max(initial_dist, 1)
    episode_reward += (1.0 if success else 0.0) + DISTANCE_REWARD_SCALE * progress
    episode_reward -= STEP_PENALTY * len(steps)

    env.close()
    return {
        "steps": steps,
        "episode_reward": episode_reward,
        "success": success,
        "length": len(steps),
        "valid_format_rate": valid_formats / max(len(steps), 1),
    }


def grpo_step(optimizer, train_env, max_steps, rollout_seed, debug=False):
    rollouts = []
    rollout_bar = tqdm(range(K_GROUP), desc="rollouts", leave=False, unit="ep")
    for k in rollout_bar:
        r = rollout_episode(train_env, max_steps, rollout_seed)
        rollouts.append(r)
        done = k + 1
        rollout_bar.set_postfix(
            rew=f"{np.mean([x['episode_reward'] for x in rollouts]):.2f}",
            succ=f"{np.mean([x['success'] for x in rollouts]):.0%}",
            fmt=f"{np.mean([x['valid_format_rate'] for x in rollouts]):.0%}",
        )

    rewards = np.array([r["episode_reward"] for r in rollouts], dtype=np.float32)
    advantages = compute_advantages(rewards)

    if debug:
        print(f"  rewards: {rewards.round(3)}")
        print(f"  advantages: {advantages.round(3)}")
        print(f"  successes: {[r['success'] for r in rollouts]}")
        if rollouts[0]["steps"]:
            print(f"  sample response: {rollouts[0]['steps'][0]['response_text']!r}")

    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    n_terms = 0

    for rollout, adv in zip(rollouts, advantages):
        adv_val = float(adv)
        n_steps = len(rollout["steps"])
        if n_steps == 0:
            continue

        for step in rollout["steps"]:
            prompt_ids = step["prompt_ids"].to(device)
            prompt_mask = step["prompt_mask"].to(device)
            response_ids = step["response_ids"].to(device)
            image = step["image"].to(device)

            log_prob, kl = sequence_log_probs_and_kl(
                model, ref_model, prompt_ids, response_ids, image, prompt_mask
            )
            step_loss = (-adv_val * log_prob + BETA_KL * kl) / (n_steps * K_GROUP)
            step_loss.backward()
            total_loss += step_loss.item()
            n_terms += 1

    if n_terms == 0:
        return {
            "loss": 0.0,
            "mean_reward": float(rewards.mean()),
            "success_rate": float(np.mean([r["success"] for r in rollouts])),
            "mean_length": float(np.mean([r["length"] for r in rollouts])),
            "valid_format_rate": float(np.mean([r["valid_format_rate"] for r in rollouts])),
        }

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return {
        "loss": total_loss,
        "mean_reward": float(rewards.mean()),
        "success_rate": float(np.mean([r["success"] for r in rollouts])),
        "mean_length": float(np.mean([r["length"] for r in rollouts])),
        "valid_format_rate": float(np.mean([r["valid_format_rate"] for r in rollouts])),
    }

## 5. Curriculum GRPO Training Loop

In [ ]:
STAGES = [
    {"env": "MiniGrid-Empty-8x8-v0",   "iters": 2, "max_steps": 50},
    {"env": "MiniGrid-Empty-10x10-v0", "iters": 2, "max_steps": 70},
    {"env": "MiniGrid-Empty-12x12-v0", "iters": 2, "max_steps": 90},
    {"env": "MiniGrid-Empty-16x16-v0", "iters": 2, "max_steps": 110},
]
N_ITERATIONS = sum(s["iters"] for s in STAGES)

EVAL_ENV = "MiniGrid-Empty-16x16-v0"
EVAL_ENVS = ["MiniGrid-Empty-8x8-v0", "MiniGrid-Empty-16x16-v0"]
FINAL_EVAL_ENVS = [
    "MiniGrid-Empty-8x8-v0",
    "MiniGrid-Empty-10x10-v0",
    "MiniGrid-Empty-12x12-v0",
    "MiniGrid-Empty-16x16-v0",
]
K_GROUP = 8
EVAL_MAX_STEPS = 110
LR = 1e-5
BETA_KL = 0.01
STEP_PENALTY = 0.01
DISTANCE_REWARD_SCALE = 0.5
PER_STEP_PROGRESS_SCALE = 0.1
TEMPERATURE = 1.5
MAX_RESPONSE_TOKENS = 48
SUCCESS_THRESHOLD = 0.5
SEED = 42
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

train_history = {
    "iteration": [],
    "stage": [],
    "train_env": [],
    "loss": [],
    "mean_reward": [],
    "rollout_success": [],
    "mean_length": [],
    "valid_format_rate": [],
    "cumulative_env_steps": [],
}
stage_eval_history = []
eval_history = {env_id: {"iteration": [], "success_rate": [], "avg_return": [], "cumulative_env_steps": []} for env_id in EVAL_ENVS}
sample_efficiency = {
    "cumulative_env_steps": [],
    "eval_16x16_success": [],
    "iteration": [],
}

global_it = 0
cumulative_env_steps = 0
stage_boundaries = [0]
last_eval_16_sr = None
steps_to_threshold = None

pbar_global = tqdm(total=N_ITERATIONS, desc="GRPO reasoning", unit="iter")

for stage_idx, stage in enumerate(STAGES):
    train_env = stage["env"]
    max_steps = stage["max_steps"]
    stage_label = env_label(train_env)

    tqdm.write(f"\n{'=' * 60}")
    tqdm.write(
        f"Stage {stage_idx + 1}/{len(STAGES)}: {stage_label} — "
        f"{stage['iters']} iters, max_steps={max_steps}"
    )
    tqdm.write(f"{'=' * 60}")

    for local_it in range(stage["iters"]):
        global_it += 1
        rollout_seed = SEED + global_it * 1000 + stage_idx * 10000 + random.randint(0, 999)
        stats = grpo_step(
            optimizer, train_env, max_steps, rollout_seed,
            debug=(stage_idx == 0 and local_it == 0),
        )

        cumulative_env_steps += int(stats["mean_length"] * K_GROUP)

        if device.type == "cuda":
            torch.cuda.empty_cache()

        train_history["iteration"].append(global_it)
        train_history["stage"].append(stage_idx)
        train_history["train_env"].append(train_env)
        train_history["cumulative_env_steps"].append(cumulative_env_steps)
        for key in ("loss", "mean_reward", "success_rate", "mean_length", "valid_format_rate"):
            mapped = "rollout_success" if key == "success_rate" else key
            train_history[mapped].append(stats[key])

        postfix = {
            "stage": f"{stage_idx + 1}/{len(STAGES)} {stage_label}",
            "loss": f"{stats['loss']:.3f}",
            "rew": f"{stats['mean_reward']:.2f}",
            "succ": f"{stats['success_rate']:.0%}",
            "fmt": f"{stats['valid_format_rate']:.0%}",
        }
        if last_eval_16_sr is not None:
            postfix["16x16"] = f"{last_eval_16_sr:.0%}"
        pbar_global.set_description(f"GRPO reasoning {stage_label}")
        pbar_global.set_postfix(postfix, refresh=True)
        pbar_global.update(1)

    stage_boundaries.append(global_it)

    tqdm.write(f"\nPost-stage {stage_idx + 1} eval...")
    eval_bar = tqdm(EVAL_ENVS, desc=f"Stage {stage_idx + 1} eval", leave=True, unit="env")
    metrics_16 = None
    for env_id in eval_bar:
        eval_steps = EVAL_MAX_STEPS if "16x16" in env_id else STAGES[stage_idx]["max_steps"]
        label = env_label(env_id)
        metrics = evaluate_policy(
            model, env_id, n_episodes=20, max_steps=eval_steps, seed=SEED,
            desc=f"  {label}", leave=False,
        )
        eval_history[env_id]["iteration"].append(global_it)
        eval_history[env_id]["success_rate"].append(metrics["success_rate"])
        eval_history[env_id]["avg_return"].append(metrics["avg_return"])
        eval_history[env_id]["cumulative_env_steps"].append(cumulative_env_steps)
        if env_id == EVAL_ENV:
            metrics_16 = metrics
            sample_efficiency["cumulative_env_steps"].append(cumulative_env_steps)
            sample_efficiency["eval_16x16_success"].append(metrics["success_rate"])
            sample_efficiency["iteration"].append(global_it)
            if steps_to_threshold is None and metrics["success_rate"] >= SUCCESS_THRESHOLD:
                steps_to_threshold = cumulative_env_steps
        eval_bar.set_postfix(**{label: f"{metrics['success_rate']:.0%}"}, refresh=True)

    last_eval_16_sr = metrics_16["success_rate"]
    stage_eval_history.append({
        "stage": stage_idx,
        "train_env": train_env,
        "iteration": global_it,
        "cumulative_env_steps": cumulative_env_steps,
        "eval_16x16": metrics_16,
    })
    tqdm.write(
        f"  Stage {stage_idx + 1} done — 16x16: "
        f"succ={metrics_16['success_rate']:.1%}, ret={metrics_16['avg_return']:.3f}, "
        f"env_steps={cumulative_env_steps}"
    )

pbar_global.close()
tqdm.write(f"\nCurriculum complete: {global_it} iterations, {cumulative_env_steps} env steps")

GRPO reasoning:   0%|          | 0/8 [00:00<?, ?iter/s]


Stage 1/4: 8x8 — 2 iters, max_steps=50


rollouts:   0%|          | 0/8 [00:00<?, ?ep/s]

  rewards: [2.39 2.37 2.   2.35 2.23 2.39 2.33 2.28]
  advantages: [ 0.798  0.635 -2.395  0.471 -0.512  0.798  0.307 -0.102]
  successes: [True, True, True, True, True, True, True, True]
  sample response: '222'


rollouts:   0%|          | 0/8 [00:00<?, ?ep/s]


Post-stage 1 eval...


Stage 1 eval:   0%|          | 0/2 [00:00<?, ?env/s]

  8x8:   0%|          | 0/20 [00:00<?, ?ep/s]

  16x16:   0%|          | 0/20 [00:00<?, ?ep/s]

## 6. Final Evaluation & Comparison with Stage 2 (GRPO action-only)

In [ ]:
print("Final GRPO reasoning evaluation")
grpo_reasoning_eval = {}
final_eval_bar = tqdm(FINAL_EVAL_ENVS, desc="Final eval", unit="env")
for env_id in final_eval_bar:
    eval_steps = EVAL_MAX_STEPS if "16x16" in env_id else 100
    label = env_label(env_id)
    grpo_reasoning_eval[env_id] = evaluate_policy(
        model, env_id, n_episodes=50, max_steps=eval_steps, seed=SEED,
        desc=f"  {label}", leave=False,
    )
    m = grpo_reasoning_eval[env_id]
    final_eval_bar.set_postfix(**{label: f"{m['success_rate']:.0%}"}, ret=f"{m['avg_return']:.2f}", refresh=True)
    tqdm.write(f"  {env_id}: success={m['success_rate']:.1%}, return={m['avg_return']:.3f}")

grpo_action_eval = grpo_action_baseline.get("eval", {}) if grpo_action_baseline else {}
grpo_action_sample_eff = None
if grpo_action_baseline:
    hist = grpo_action_baseline.get("stage_eval_history", [])
    for rec in hist:
        sr = rec.get("eval_16x16", {}).get("success_rate", 0)
        if sr >= SUCCESS_THRESHOLD:
            grpo_action_sample_eff = rec.get("cumulative_env_steps")
            if grpo_action_sample_eff is None:
                grpo_action_sample_eff = rec.get("iteration", 0) * K_GROUP * 30
            break

print("\n=== Comparison: GRPO action-only (Stage 2) vs GRPO reasoning (Stage 3) ===")
print(f"{'Method':<18} {'Env':<8} {'Success':>10} {'Return':>10}")
print("-" * 50)
for env_id in FINAL_EVAL_ENVS:
    label = env_label(env_id)
    act = grpo_action_eval.get(env_id, {})
    act_sr = act.get("success_rate", float("nan"))
    act_ret = act.get("avg_return", float("nan"))
    reas = grpo_reasoning_eval[env_id]
    print(f"{'GRPO action':<18} {label:<8} {act_sr:>9.0%} {act_ret:>10.3f}")
    print(f"{'GRPO reasoning':<18} {label:<8} {reas['success_rate']:>9.0%} {reas['avg_return']:>10.3f}")
    if not np.isnan(act_sr):
        print(f"{'Delta':<18} {label:<8} {reas['success_rate'] - act_sr:>+9.0%}")
    print()

print("=== Sample Efficiency (16x16, threshold={:.0%}) ===".format(SUCCESS_THRESHOLD))
print(f"  GRPO reasoning: env_steps={steps_to_threshold or 'not reached'}")
print(f"  GRPO action:    env_steps={grpo_action_sample_eff or 'not reached / no baseline'}")
if steps_to_threshold and grpo_action_sample_eff:
    delta = steps_to_threshold - grpo_action_sample_eff
    print(f"  Delta: {delta:+d} env steps (negative = reasoning faster)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
stage_colors = ["#4CAF50", "#8BC34A", "#FFC107", "#FF5722"]
stage_names = [env_label(s["env"]) for s in STAGES]

ax = axes[0, 0]
for i in range(len(STAGES)):
    ax.axvspan(stage_boundaries[i] + 0.5, stage_boundaries[i + 1] + 0.5,
               alpha=0.15, color=stage_colors[i], label=stage_names[i])
ax.plot(train_history["iteration"], train_history["mean_reward"], alpha=0.8, color="black")
ax.set_title("GRPO Reasoning: Mean Group Reward")
ax.set_xlabel("Global Iteration")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for i in range(len(STAGES)):
    ax.axvspan(stage_boundaries[i] + 0.5, stage_boundaries[i + 1] + 0.5, alpha=0.15, color=stage_colors[i])
ax.plot(train_history["iteration"], train_history["rollout_success"], alpha=0.8, color="orange", label="success")
ax.plot(train_history["iteration"], train_history["valid_format_rate"], alpha=0.8, color="#9C27B0", label="valid format")
ax.set_title("Rollout Success & Format Compliance")
ax.set_xlabel("Global Iteration")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
stage_iters = [rec["iteration"] for rec in stage_eval_history]
stage_16_sr = [rec["eval_16x16"]["success_rate"] for rec in stage_eval_history]
if grpo_action_baseline and grpo_action_baseline.get("stage_eval_history"):
    act_iters = [r["iteration"] for r in grpo_action_baseline["stage_eval_history"]]
    act_sr = [r["eval_16x16"]["success_rate"] for r in grpo_action_baseline["stage_eval_history"]]
    ax.plot(act_iters, act_sr, marker="s", color="#4CAF50", linestyle="--", label="GRPO action (Stage 2)")
ax.plot(stage_iters, stage_16_sr, marker="o", color="#2196F3", linewidth=2, label="GRPO reasoning (Stage 3)")
ax.axhline(SUCCESS_THRESHOLD, color="gray", linestyle=":", label=f"threshold {SUCCESS_THRESHOLD:.0%}")
ax.set_title("16x16 Success After Each Stage")
ax.set_xlabel("Global Iteration")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
if sample_efficiency["cumulative_env_steps"]:
    ax.plot(sample_efficiency["cumulative_env_steps"], sample_efficiency["eval_16x16_success"],
            marker="o", color="#2196F3", label="GRPO reasoning")
ax.axhline(SUCCESS_THRESHOLD, color="gray", linestyle=":")
ax.set_title("Sample Efficiency: 16x16 Success vs Env Steps")
ax.set_xlabel("Cumulative Training Env Steps")
ax.set_ylabel("16x16 Success Rate")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
x = np.arange(len(FINAL_EVAL_ENVS))
width = 0.35
act_rates = [grpo_action_eval.get(e, {}).get("success_rate", 0.0) for e in FINAL_EVAL_ENVS]
reas_rates = [grpo_reasoning_eval[e]["success_rate"] for e in FINAL_EVAL_ENVS]
labels = [env_label(e) for e in FINAL_EVAL_ENVS]
ax.bar(x - width / 2, act_rates, width, label="GRPO action", color="#4CAF50")
ax.bar(x + width / 2, reas_rates, width, label="GRPO reasoning", color="#2196F3")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_title("Final Success Rate: Stage 2 vs Stage 3")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

ax = axes[1, 2]
deltas = [reas_rates[i] - act_rates[i] for i in range(len(FINAL_EVAL_ENVS))]
colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars = ax.bar(labels, deltas, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Success Rate Delta (Reasoning − Action)")
ax.set_ylim(-1.05, 1.05)
ax.grid(True, alpha=0.3, axis="y")
for bar, d in zip(bars, deltas):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (0.02 if d >= 0 else -0.06),
            f"{d:+.0%}", ha="center", fontsize=10)

plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / "grpo_reasoning_learning_curves.png", dpi=150)
plt.show()

In [ ]:
model.save_pretrained(str(OUTPUT_DIR / "checkpoint"))

results = {
    "method": "GRPO_reasoning",
    "config": {
        "init_checkpoint": str(INIT_CHECKPOINT),
        "init_source": INIT_SOURCE,
        "prompt": PROMPT,
        "output_format": "Thought: <one sentence>\nAction: <0|1|2>",
        "stages": STAGES,
        "k_group": K_GROUP,
        "n_iterations": N_ITERATIONS,
        "lr": LR,
        "beta_kl": BETA_KL,
        "max_response_tokens": MAX_RESPONSE_TOKENS,
        "temperature": TEMPERATURE,
        "success_threshold": SUCCESS_THRESHOLD,
    },
    "train_history": train_history,
    "stage_eval_history": stage_eval_history,
    "eval_history": eval_history,
    "sample_efficiency": sample_efficiency,
    "steps_to_threshold_16x16": steps_to_threshold,
    "eval": grpo_reasoning_eval,
    "grpo_action_baseline_eval": grpo_action_eval,
    "grpo_action_baseline_steps_to_threshold": grpo_action_sample_eff,
    "stage_boundaries": stage_boundaries,
}
with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Checkpoint: {OUTPUT_DIR / 'checkpoint'}")
print(f"Results:    {OUTPUT_DIR / 'results.json'}")
print(f"Plot:       {OUTPUT_DIR / 'grpo_reasoning_learning_curves.png'}")